In [ ]:
import sqlite3

import mlflow
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# same experiment as the OpenAI agent, so we can compare the traces side by side
mlflow.set_experiment("music_agent")
mlflow.langchain.autolog()

In [ ]:
# same tool as before, now as a LangChain tool
db = sqlite3.connect("file:../data/music.db?mode=ro", uri=True, check_same_thread=False)


@tool
def run_sql(query: str) -> str:
    """Run a SQL query on the music database (SQLite)."""
    try:
        return str(db.execute(query).fetchall())
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# the loop we wrote by hand is now one function call
# no @mlflow.trace needed: autolog captures the agent, the LLM calls and the tools
agent = create_agent(
    "openai:gpt-6-luna",
    tools=[run_sql],
    system_prompt="You answer questions about a music store. Look at the tables and their columns before writing a query.",
)

In [ ]:
def ask(question: str) -> str:
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return response["messages"][-1].text

In [ ]:
ask("Which genre on average has the longest tracks?")

In [ ]:
ask("List the 5 artists with the most albums.")

In [ ]:
ask("Who is the best-selling artist, and what is their best-selling album?")